In [1]:
import os
import pandas as pd
import numpy as np
import cv2
import tensorflow as tf
from pathlib import Path
import random
from sklearn.model_selection import train_test_split


I0000 00:00:1777362761.504386    5020 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777362761.614266    5020 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777362773.551073    5020 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
DATASET_DIR  = "frames"
OUTPUT_DIR   = "tfrecords"
NUM_FRAMES   = 16
IMG_SIZE     = 224
TRAIN_SPLIT  = 0.8
VAL_SPLIT    = 0.1
TEST_SPLIT   = 0.1
SHARD_SIZE   = 100

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/train", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/val",   exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/test",  exist_ok=True)

In [3]:
def load_and_resize_frame(frame_path):

    raw   = tf.io.read_file(frame_path)
    image = tf.image.decode_jpeg(raw, channels=3)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32) / 255.0
    return image


def sample_frames(frame_paths, n=NUM_FRAMES):

    total = len(frame_paths)
    if total >= n:
        indices = [int(i * total / n) for i in range(n)]
        return [frame_paths[i] for i in indices]
    else:

        return frame_paths + [frame_paths[-1]] * (n - total)


def serialize_video(video_folder, label):
    frame_paths = sorted([
        str(p) for p in Path(video_folder).glob("*.jpg")
    ])
    frame_paths += sorted([
        str(p) for p in Path(video_folder).glob("*.png")
    ])
    frame_paths = sorted(set(frame_paths))

    if len(frame_paths) == 0:
        print(f"  ⚠ Skipping empty folder: {video_folder}")
        return None

    sampled = sample_frames(frame_paths, NUM_FRAMES)

    frames = []
    for fp in sampled:
        img = load_and_resize_frame(fp).numpy()
        frames.append(img)

    stacked   = tf.stack(frames, axis=0)
    raw_bytes = tf.io.serialize_tensor(stacked).numpy()

    feature = {
        "frames": tf.train.Feature(
            bytes_list=tf.train.BytesList(value=[raw_bytes])
        ),
        "label": tf.train.Feature(
            int64_list=tf.train.Int64List(value=[label])
        ),
        "video_name": tf.train.Feature(
            bytes_list=tf.train.BytesList(
                value=[os.path.basename(video_folder).encode()]
            )
        ),
    }
    return tf.train.Example(
        features=tf.train.Features(feature=feature)
    ).SerializeToString()




def collect_videos(dataset_dir):
    videos = []
    for label, split_name in [(0, "real"), (1, "fake")]:
        split_dir = Path(dataset_dir) / split_name
        for video_folder in sorted(split_dir.iterdir()):
            if video_folder.is_dir():
                videos.append((str(video_folder), label))
    random.shuffle(videos)
    return videos



def split_videos(videos):
    n       = len(videos)
    n_train = int(n * TRAIN_SPLIT)
    n_val   = int(n * VAL_SPLIT)
    train   = videos[:n_train]
    val     = videos[n_train:n_train + n_val]
    test    = videos[n_train + n_val:]
    return train, val, test



def write_tfrecords(video_list, output_subdir, split_name):
    print(f"\n Writing {split_name} — {len(video_list)} videos")
    shard_idx  = 0
    writer     = None
    written    = 0
    skipped    = 0

    for i, (video_folder, label) in enumerate(video_list):
        if i % SHARD_SIZE == 0:
            if writer:
                writer.close()
            shard_path = os.path.join(
                output_subdir,
                f"{split_name}_shard_{shard_idx:04d}.tfrecord"
            )
            writer    = tf.io.TFRecordWriter(shard_path)
            shard_idx += 1
            print(f"  → Shard {shard_idx}: {shard_path}")

        serialized = serialize_video(video_folder, label)

        if serialized:
            writer.write(serialized)
            written += 1
        else:
            skipped += 1

    if writer:
        writer.close()

    print(f"  ✓ Done — written: {written}, skipped: {skipped}")



In [4]:
print("Collecting video folders...")
videos = collect_videos(DATASET_DIR)
print(f"Total videos found: {len(videos)}")

real_count = sum(1 for _, l in videos if l == 0)
fake_count = sum(1 for _, l in videos if l == 1)
print(f"  Real: {real_count}  |  Fake: {fake_count}")

train, val, test = split_videos(videos)
print(f"  Train: {len(train)}  Val: {len(val)}  Test: {len(test)}")

write_tfrecords(train, f"{OUTPUT_DIR}/train", "train")
write_tfrecords(val,   f"{OUTPUT_DIR}/val",   "val")
write_tfrecords(test,  f"{OUTPUT_DIR}/test",  "test")

print("\n✅ TFRecord conversion complete!")
print(f"   Saved to: {OUTPUT_DIR}")

Total videos found: 15562
  Real: 7781  |  Fake: 7781
  Train: 12449  Val: 1556  Test: 1557

 Writing train — 12449 videos
  → Shard 1: tfrecords/train/train_shard_0000.tfrecord


W0000 00:00:1777362838.852641    5020 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


  → Shard 2: tfrecords/train/train_shard_0001.tfrecord
  → Shard 3: tfrecords/train/train_shard_0002.tfrecord
  → Shard 4: tfrecords/train/train_shard_0003.tfrecord
  → Shard 5: tfrecords/train/train_shard_0004.tfrecord
  → Shard 6: tfrecords/train/train_shard_0005.tfrecord
  → Shard 7: tfrecords/train/train_shard_0006.tfrecord
  → Shard 8: tfrecords/train/train_shard_0007.tfrecord
  → Shard 9: tfrecords/train/train_shard_0008.tfrecord
  → Shard 10: tfrecords/train/train_shard_0009.tfrecord
  → Shard 11: tfrecords/train/train_shard_0010.tfrecord
  → Shard 12: tfrecords/train/train_shard_0011.tfrecord
  → Shard 13: tfrecords/train/train_shard_0012.tfrecord
  → Shard 14: tfrecords/train/train_shard_0013.tfrecord
  → Shard 15: tfrecords/train/train_shard_0014.tfrecord
  → Shard 16: tfrecords/train/train_shard_0015.tfrecord
  → Shard 17: tfrecords/train/train_shard_0016.tfrecord
  → Shard 18: tfrecords/train/train_shard_0017.tfrecord
  → Shard 19: tfrecords/train/train_shard_0018.tfrecord
